# 09 — AI Alignment: IVE-Weighted Utility

This notebook explores how the Identified Victim Effect challenges standard utilitarian reasoning in AI alignment contexts.

**Core question:** If AI systems use utilitarian aggregation for moral decisions, they treat welfare as perfectly fungible. The IVE suggests humans weight identified individuals more heavily. Should AI systems adopt IVE-weighting, or is this a bias to correct?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

from ive.alignment import (
    Individual, Scenario,
    utilitarian_utility, ive_weighted_utility, ive_weighted_utility_with_floor,
    compare_aggregations,
    repugnant_conclusion, trolley_identified_statistical,
    resource_allocation, scope_insensitivity,
    run_all_scenarios,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

## 1. IVE-Weighted Utility

**Standard utilitarian:** $U = \sum_i u_i \cdot n_i$

**IVE-weighted:** $U_{IVE} = \sum_i w_i \cdot u_i \cdot n_i$ where $w_i = 1 + c \cdot \text{identity}_i$

When coupling $c = 0$, IVE reduces to standard utilitarianism.

In [ ]:
# Demonstration
identified_person = Individual(utility=10.0, identity_level=1.0, label='Named child')
anonymous_person = Individual(utility=10.0, identity_level=0.0, label='Statistical victim')

coupling_values = np.linspace(0, 2, 20)
id_utils = [ive_weighted_utility([identified_person], c) for c in coupling_values]
anon_utils = [ive_weighted_utility([anonymous_person], c) for c in coupling_values]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(coupling_values, id_utils, 'r-o', label='Identified person', linewidth=2, markersize=4)
ax.plot(coupling_values, anon_utils, 'b-s', label='Anonymous person', linewidth=2, markersize=4)
ax.axvline(0.65, color='gray', linestyle=':', label='Fitted coupling (0.65)')
ax.set_xlabel('IVE Coupling (c)')
ax.set_ylabel('Weighted Utility')
ax.set_title('IVE-Weighted Utility: Same welfare, different weights')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/09_ive_utility_demo.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Repugnant Conclusion (Parfit, 1984)

**Option A:** 10 identified, happy people (utility=100 each)

**Option B:** 2000 anonymous, barely-living people (utility=1 each)

Utilitarianism prefers B (total=2000 > 1000). Does IVE-weighting prefer A?

In [ ]:
rc = repugnant_conclusion()
print(rc.description)
print()

df = compare_aggregations(rc, coupling_values=[0.0, 0.3, 0.65, 1.0, 1.5, 2.0])
df.style.set_caption('Repugnant Conclusion: utilitarian vs IVE-weighted')

In [ ]:
couplings = np.linspace(0, 3, 50)
util_a = [ive_weighted_utility(rc.options['A_small_happy'], c) for c in couplings]
util_b = [ive_weighted_utility(rc.options['B_large_barely'], c) for c in couplings]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(couplings, util_a, 'r-', label='A: 10 happy, identified', linewidth=2)
ax.plot(couplings, util_b, 'b-', label='B: 2000 barely-living, anonymous', linewidth=2)

# Find crossover
for i in range(len(couplings)-1):
    if util_a[i] < util_b[i] and util_a[i+1] >= util_b[i+1]:
        ax.axvline(couplings[i], color='green', linestyle='--', label=f'Crossover ~{couplings[i]:.2f}')
        break

ax.set_xlabel('IVE Coupling')
ax.set_ylabel('Weighted Total Utility')
ax.set_title('Repugnant Conclusion: When does IVE-weighting prefer the small happy population?')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/09_repugnant_conclusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Trolley Problem with Identified Victim

In [ ]:
sc_id = trolley_identified_statistical(victim_identified=True)
sc_anon = trolley_identified_statistical(victim_identified=False)

couplings = np.linspace(0, 3, 50)
# Net utility of diverting vs not diverting
net_id = [ive_weighted_utility(sc_id.options['divert'], c) - ive_weighted_utility(sc_id.options['do_nothing'], c) for c in couplings]
net_anon = [ive_weighted_utility(sc_anon.options['divert'], c) - ive_weighted_utility(sc_anon.options['do_nothing'], c) for c in couplings]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(couplings, net_id, 'r-', label='Identified side-track victim', linewidth=2)
ax.plot(couplings, net_anon, 'b-', label='Anonymous side-track victim', linewidth=2)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.fill_between(couplings, 0, 10, alpha=0.05, color='green', label='Divert preferred')
ax.fill_between(couplings, -10, 0, alpha=0.05, color='red', label='Do nothing preferred')

ax.set_xlabel('IVE Coupling')
ax.set_ylabel('Net Utility (divert - do nothing)')
ax.set_title('Trolley Problem: Does identifying the victim change the decision?')
ax.set_ylim(-3, 8)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../figures/09_trolley.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Resource Allocation: Identifiability Bias

In [ ]:
ra = resource_allocation()
print(ra.description)

df = compare_aggregations(ra, coupling_values=[0.0, 0.3, 0.65, 1.0, 2.0, 3.0])
df.style.set_caption('Resource Allocation: concentrated (identified) vs distributed (anonymous)')

In [ ]:
couplings = np.linspace(0, 5, 80)
util_conc = [ive_weighted_utility(ra.options['A_concentrated'], c) for c in couplings]
util_dist = [ive_weighted_utility(ra.options['B_distributed'], c) for c in couplings]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(couplings, util_conc, 'r-', label='A: 1 identified, benefit=80', linewidth=2)
ax.plot(couplings, util_dist, 'b-', label='B: 100 anonymous, benefit=5', linewidth=2)

# Find crossover
for i in range(len(couplings)-1):
    if util_conc[i] < util_dist[i] and util_conc[i+1] >= util_dist[i+1]:
        ax.axvline(couplings[i], color='green', linestyle='--', label=f'Crossover ~{couplings[i]:.1f}')
        break

ax.set_xlabel('IVE Coupling')
ax.set_ylabel('Weighted Total Utility')
ax.set_title('Resource Allocation: The IVE as identifiability bias')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/09_resource_allocation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Scope Insensitivity

In [ ]:
results = run_all_scenarios(coupling_values=[0.0, 0.3, 0.65, 1.0, 1.5])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, key, title in [(axes[0], 'scope_anonymous', 'Anonymous Victims'),
                        (axes[1], 'scope_identified', 'Identified Victims')]:
    df = results[key]
    for c, grp in df.groupby('coupling'):
        ax.plot(grp['n'], grp['ive_utility'], 'o-', label=f'c={c}', linewidth=1.5)
    ax.set_xscale('log')
    ax.set_xlabel('Number of Victims')
    ax.set_ylabel('IVE-Weighted Utility')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Scope Insensitivity: IVE utility vs group size', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/09_scope_insensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. The Dark Side: Photogenic Victim Bias

The IVE creates a systematic bias toward identifiable victims — those with names, faces, and stories. This can lead to:

- **Media bias**: Photogenic victims receive disproportionate resources
- **Proximity bias**: Local victims weighted more than distant ones
- **Narrative bias**: Single compelling story > aggregate statistics

For AI alignment, this means IVE-weighting could perpetuate or amplify existing biases in charitable giving and policy decisions.

In [ ]:
# Photogenic victim scenario: one identified child vs many anonymous
photogenic = Scenario(
    name='Photogenic Victim Bias',
    description='Fund medical treatment for 1 named child vs vaccines for 1000 anonymous children',
    options={
        'named_child': [
            Individual(utility=50.0, identity_level=1.0, group_size=1,
                      label='Named child with photo')
        ],
        'vaccine_program': [
            Individual(utility=2.0, identity_level=0.0, group_size=1000,
                      label='1000 anonymous children')
        ],
    },
)

df = compare_aggregations(photogenic, coupling_values=[0.0, 0.3, 0.65, 1.0, 2.0])
print('Utilitarian: vaccine program (2000) >> named child (50)')
print('But with high coupling, IVE-weighting can reverse this:')
df.style.set_caption('Photogenic Victim Bias')

## 7. Full Coupling Sweep Across All Scenarios

In [ ]:
couplings = np.linspace(0, 3, 50)

scenarios = {
    'Repugnant Conclusion': repugnant_conclusion(),
    'Trolley (identified victim)': trolley_identified_statistical(victim_identified=True),
    'Resource Allocation': resource_allocation(),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, sc) in zip(axes, scenarios.items()):
    for opt_name, inds in sc.options.items():
        utils = [ive_weighted_utility(inds, c) for c in couplings]
        ax.plot(couplings, utils, linewidth=2, label=opt_name)
    ax.axvline(0.65, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Coupling')
    ax.set_ylabel('IVE Utility')
    ax.set_title(name)
    ax.legend(fontsize=8)

plt.suptitle('IVE-Weighted Utility Across Coupling Values', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/09_coupling_sweep_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Link to Neural Model

The coupling parameter in the alignment module maps directly to the `identity_affect_coupling` parameter in the neural model:

- **Neural level**: TPJ identity precision → Insula affect amplification
- **Behavioral level**: Identified victims receive more help
- **Alignment level**: Identified individuals receive higher utility weight

The fitted coupling value (0.65) from behavioral data suggests humans apply moderate IVE-weighting — not pure utilitarianism (c=0), but not extreme identifiability bias either.

In [ ]:
# What does coupling=0.65 imply for the alignment scenarios?
c_fitted = 0.65

for name, sc in scenarios.items():
    utils = {opt: ive_weighted_utility(inds, c_fitted) for opt, inds in sc.options.items()}
    preferred = max(utils, key=utils.get)
    print(f'{name}:')
    for opt, u in utils.items():
        marker = ' <-- preferred' if opt == preferred else ''
        print(f'  {opt}: {u:.1f}{marker}')
    print()

## Summary

**Key alignment insights from the IVE model:**

1. **IVE-weighting avoids the Repugnant Conclusion** by making identified welfare non-fungible — a small happy identified population can be preferred over a large barely-living anonymous one.

2. **But IVE-weighting introduces identifiability bias** — resources flow toward photogenic, named victims at the expense of statistically larger but anonymous populations.

3. **The coupling parameter controls the trade-off**: c=0 is pure utilitarianism (scope-sensitive but repugnant-conclusion-prone), high c is strong IVE (repugnant-conclusion-resistant but biased).

4. **For AI alignment**, the moderate coupling observed in humans (c~0.65) suggests a middle path — but whether AI systems *should* adopt this weighting depends on normative commitments about the moral status of identifiability.

5. **Neural grounding** provides a mechanism: the TPJ-Insula coupling parameter isn't just a fit parameter; it has a specific neural implementation that can be measured with fMRI and modulated with TMS.